## build_dim_date
Rebuilds `gold.dim_date` by carrying `silver.dim_date` **verbatim** (`date_key` is the deterministic yyyymmdd INT). Full rebuild via `INSERT OVERWRITE` (design §2.2), preserving the G0 PK + COMMENTs. StepLog + transform_detail_log. Spec: `gold_layer_design.md` §3.2.

In [ ]:
%run "../libs/notebook_init"

In [ ]:
STEP_SEQUENCE = 2                       # position owned by the orchestrator (G4)
SOURCE_TABLE  = f"{SILVER}.dim_date"
TARGET_TABLE  = f"{GOLD}.dim_date"

# Carried verbatim, IN G0 COLUMN ORDER (gold_ddl.py). Audit ts set fresh at write.
CARRY_COLS = [
    "date_key", "full_date", "year", "quarter", "month",
    "month_start", "quarter_start", "is_month_end", "is_quarter_end",
]

In [ ]:
nb = Utils.get_notebook_context(dbutils)
step = StepLog(
    spark, AUDIT, dbutils,
    pipeline_run_id = PIPELINE_RUN_ID,
    step_sequence   = STEP_SEQUENCE,
    notebook_folder = nb["notebook_folder"],
    notebook_name   = nb["notebook_name"],
    layer           = "gold",
    target_table    = TARGET_TABLE,
)
print(f"build_dim_date: step_log_id={step.step_log_id}")

In [ ]:
# Carry Silver dim_date verbatim; set fresh Gold audit timestamps.
try:
    src = spark.table(SOURCE_TABLE)
    rows_read = src.count()
    staged = src.select(
        *[F.col(col_name) for col_name in CARRY_COLS],
        F.current_timestamp().alias("inserted_ts"),
        F.current_timestamp().alias("updated_ts"),
    )
    staged.createOrReplaceTempView("gold_dim_date_staging")
    step.rows_read = rows_read
    print(f"build_dim_date: read {rows_read:,} Silver dim_date rows")
except Exception as e:
    step.fail(e); raise

In [ ]:
# Full rebuild via INSERT OVERWRITE (design §2.2) — preserves G0 schema/PK/COMMENTs.
transform_started = datetime.now(timezone.utc)
try:
    spark.sql(f"INSERT OVERWRITE TABLE {TARGET_TABLE} SELECT * FROM gold_dim_date_staging")

    post_count = spark.table(TARGET_TABLE).count()
    if post_count != step.rows_read:
        raise AssertionError(
            f"[{TARGET_TABLE}] Row-count mismatch: carried {step.rows_read:,} from "
            f"{SOURCE_TABLE}, table now has {post_count:,}."
        )
    step.rows_written = post_count
    transform_detail_log_insert(
        spark, AUDIT, PIPELINE_RUN_ID, step.step_log_id, SOURCE_TABLE, TARGET_TABLE,
        status=STATUS_SUCCEEDED, started_timestamp=transform_started, rows_read=step.rows_read,
        rows_written=post_count, rows_inserted=post_count, ended_timestamp=datetime.now(timezone.utc))
    step.succeed()
    print(f"build_dim_date: wrote {post_count:,} rows to {TARGET_TABLE}")
except Exception as e:
    transform_detail_log_insert(
        spark, AUDIT, PIPELINE_RUN_ID, step.step_log_id, SOURCE_TABLE, TARGET_TABLE,
        status=STATUS_FAILED, started_timestamp=transform_started, rows_read=step.rows_read,
        error_message=f"{type(e).__name__}: {e}", ended_timestamp=datetime.now(timezone.utc))
    step.fail(e); raise